In [49]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [50]:
documents = [file.parse() for file in files]

Q1. How many lesson pages   

In [51]:
len(documents)

72

Q2. Indexing and searching      
Index the documents with minsearch - make content a text field and filename a keyword field.

In [52]:
print(documents[0].keys())

dict_keys(['content', 'filename'])


In [53]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

question="How does the agentic loop keep calling the model until it stops?"
search_results = index.search(question)

In [54]:
(search_results[0].keys())

dict_keys(['content', 'filename'])

In [55]:
print(search_results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


Q3. RAG
Build a RAG assistant on top of this data.

Run ```wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py``` 

In [56]:
from openai import OpenAI
import os
openai_client=OpenAI()

In [57]:
from rag_helper import RAGBase
query = "How does the agentic loop keep calling the model until it stops?"
rag_system = RAGBase(index=index, llm_client=openai_client)
answer, usage = rag_system.rag(query=query)


In [58]:
print(answer)
print(usage.input_tokens)

It keeps calling the model in a `while True` loop. After each response, the code checks whether the model returned any `function_call` items:

- if yes, it runs the tool, appends the tool output to the message history, and loops again;
- if no function calls are returned, it breaks out of the loop.

So the stop condition is: **no function calls in the current turn**.
7131


Q4. Chunking

In [59]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(documents)) #72
print(len(chunks)) #295

72
295


Q5. RAG with chunking

In [60]:
index_chunking = Index(text_fields=["content"], keyword_fields=["filename"])
index_chunking.fit(chunks)

In [61]:
rag_system = RAGBase(index=index_chunking, llm_client=openai_client)
answer, usage = rag_system.rag(query=query)

In [62]:
print(usage.input_tokens)

2314


Q6. Turning it into an agent    

# uv add toyaikit

In [63]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [79]:
search_calls = 0
def search(query: str)-> list[dict]:
    """
    Search the lesson contents to match the given query.

    Use the search to find relevant lesson materials to answer the student's query question.
    """
    global search_calls 
    search_calls += 1
    return index_chunking.search(query, num_results=5)

In [80]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [81]:
INSTRUCTIONS = '''
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
'''.strip()

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [82]:
QUERY ="How does the agentic loop work, and how is it different from plain RAG?"

result = runner.loop(
    prompt=QUERY,
    callback=callback
)

-> Response received


-> Response received


In [83]:
print(search_calls)

3
